# Draft 03 — Modélisation et comparaison

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import GridSearchCV

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from preprocessing import (
    TARGET_COLUMN,
    add_engineered_features,
    build_preprocessor,
    get_feature_columns,
    load_data,
)
from train import evaluate_model, get_models, save_best_model
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

df = add_engineered_features(load_data(ROOT / 'data' / 'diabetes.csv'))
feature_columns = get_feature_columns(include_engineered=True)
X = df[feature_columns]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
preprocessor = build_preprocessor(feature_columns)

In [ ]:
results = []
fitted = {}
for name, model in get_models().items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_proba)
    metrics['cv_f1_mean'] = cross_val_score(pipe, X, y, cv=5, scoring='f1').mean()
    metrics['model'] = name
    results.append(metrics)
    fitted[name] = pipe

results_df = pd.DataFrame(results).set_index('model').round(4)
results_df

In [ ]:
best_name = results_df['f1'].idxmax()
best_model = fitted[best_name]
save_best_model(best_model, ROOT / 'models' / 'best_model.joblib')
print(f'Meilleur modèle (F1 test): {best_name}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=axes[0])
axes[0].set_title(f'Matrice de confusion — {best_name}')
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
axes[1].set_title('Courbe ROC — meilleur modèle')
plt.tight_layout()
plt.show()

## GridSearchCV sur Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42)),
])
search = GridSearchCV(
    rf_pipe,
    param_grid={
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 5, 10],
    },
    cv=5,
    scoring='f1',
)
search.fit(X_train, y_train)
print('Meilleurs hyperparamètres:', search.best_params_)
print('F1 CV:', round(search.best_score_, 4))